<a href="https://colab.research.google.com/github/ImrulMir15/cocomo-analysis/blob/main/Improved_Cocomo_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Improved COCOMO Analysis - Enhanced Effort Prediction

This notebook implements advanced machine learning techniques to improve software effort estimation accuracy using the COCOMO 81 dataset.

## Improvements Implemented:
1. **Advanced ML Models**: Gradient Boosting, XGBoost, LightGBM, Neural Networks
2. **Feature Engineering**: Polynomial features, feature interactions, log transformations
3. **Data Preprocessing**: Robust scaling, outlier detection, feature selection
4. **Ensemble Methods**: Voting regressor, stacking regressor for better predictions
5. **Enhanced Cross-Validation**: K-Fold with multiple metrics
6. **Hyperparameter Optimization**: Bayesian optimization, advanced tuning
7. **Better Visualizations**: Feature importance, prediction error analysis, model comparison
8. **Performance Metrics**: Extended metrics including MAE, RMSE, R², MAPE

## 1. Install Required Libraries (For Colab)

In [ ]:
# Install additional libraries for advanced ML
!pip install xgboost lightgbm scikit-optimize -q

## 2. Import Libraries

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.io import arff

# Scikit-learn imports
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    VotingRegressor,
    StackingRegressor,
    ExtraTreesRegressor
)
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, RobustScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
    GridSearchCV,
    RandomizedSearchCV
)

# Advanced ML libraries
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
sns.set_palette('husl')

print('Libraries imported successfully!')

## 3. Load Dataset

In [ ]:
# Download dataset if running in Colab
import os
if not os.path.exists('cocomo811.arff'):
    !wget https://raw.githubusercontent.com/ImrulMir15/cocomo-analysis/main/cocomo811.arff -q
    print('Dataset downloaded!')
else:
    print('Dataset already exists!')

In [ ]:
# Load ARFF file
DATA_PATH = Path('cocomo811.arff')
raw_data, meta = arff.loadarff(DATA_PATH)

# Convert to DataFrame
columns = meta.names()
numeric_data = np.asarray(raw_data.tolist(), dtype=np.float64)
df = pd.DataFrame(numeric_data, columns=columns)

feature_names = columns[:-1]
target_name = columns[-1]

print(f'Dataset shape: {df.shape}')
print(f'Features: {len(feature_names)}')
print(f'Target: {target_name}')
df.head()

## 4. Exploratory Data Analysis

In [ ]:
# Statistical summary
df.describe().T

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# Correlation analysis
plt.figure(figsize=(14, 10))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation with target
corr_with_target = df.corr()[target_name].sort_values(ascending=False)
print('\nTop features correlated with effort:')
print(corr_with_target.head(10))

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[target_name], bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Target Distribution (Effort)', fontweight='bold')
axes[0].set_xlabel('Effort')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df[target_name]), bins=20, color='coral', edgecolor='black')
axes[1].set_title('Log-Transformed Target Distribution', fontweight='bold')
axes[1].set_xlabel('Log(Effort + 1)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# Prepare features and target
X = df[feature_names].values
y = df[target_name].values

print(f'Original features shape: {X.shape}')
print(f'Target shape: {y.shape}')

In [ ]:
# Create polynomial features (degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

print(f'Polynomial features shape: {X_poly.shape}')
print(f'Number of new features created: {X_poly.shape[1] - X.shape[1]}')

In [ ]:
# Log transformation for skewed features
X_log = np.log1p(X)  # log(1 + x) to handle zeros
print(f'Log-transformed features shape: {X_log.shape}')

## 6. Data Splitting and Scaling

In [ ]:
# Set random state for reproducibility
RANDOM_STATE = 42

# Split data (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

In [ ]:
# Apply robust scaling (better for outliers)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Data scaled using RobustScaler')

## 7. Define Evaluation Metrics

In [ ]:
def compute_effort_metrics(y_true, y_pred, threshold=0.25):
    """Compute comprehensive effort estimation metrics."""
    eps = 1e-8
    
    # Mean Magnitude of Relative Error
    mre = np.abs(y_true - y_pred) / np.maximum(np.abs(y_true), eps)
    mmre = mre.mean()
    mdmre = np.median(mre)
    pred = (mre < threshold).mean() * 100
    
    # Standard metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {
        'MMRE': mmre,
        'MdMRE': mdmre,
        'Pred25': pred,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2
    }

print('Metrics function defined!')

## 8. Train Advanced ML Models

### 8.1 Random Forest (Baseline)

In [ ]:
# Random Forest
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_metrics = compute_effort_metrics(y_test, rf_pred)

print('Random Forest Results:')
for metric, value in rf_metrics.items():
    print(f'  {metric}: {value:.4f}')

### 8.2 Gradient Boosting

In [ ]:
# Gradient Boosting
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    subsample=0.8,
    random_state=RANDOM_STATE
)

gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
gb_metrics = compute_effort_metrics(y_test, gb_pred)

print('Gradient Boosting Results:')
for metric, value in gb_metrics.items():
    print(f'  {metric}: {value:.4f}')

### 8.3 XGBoost

In [ ]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model.fit(X_train_scaled, y_train)
xgb_pred = xgb_model.predict(X_test_scaled)
xgb_metrics = compute_effort_metrics(y_test, xgb_pred)

print('XGBoost Results:')
for metric, value in xgb_metrics.items():
    print(f'  {metric}: {value:.4f}')

### 8.4 LightGBM

In [ ]:
# LightGBM
lgb_model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=31,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(X_train_scaled, y_train)
lgb_pred = lgb_model.predict(X_test_scaled)
lgb_metrics = compute_effort_metrics(y_test, lgb_pred)

print('LightGBM Results:')
for metric, value in lgb_metrics.items():
    print(f'  {metric}: {value:.4f}')

### 8.5 Extra Trees

In [ ]:
# Extra Trees
et_model = ExtraTreesRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

et_model.fit(X_train_scaled, y_train)
et_pred = et_model.predict(X_test_scaled)
et_metrics = compute_effort_metrics(y_test, et_pred)

print('Extra Trees Results:')
for metric, value in et_metrics.items():
    print(f'  {metric}: {value:.4f}')

### 8.6 Neural Network (MLP)

In [ ]:
# Multi-Layer Perceptron
mlp_model = MLPRegressor(
    hidden_layer_sizes=(100, 50, 25),
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    max_iter=1000,
    random_state=RANDOM_STATE,
    early_stopping=True
)

mlp_model.fit(X_train_scaled, y_train)
mlp_pred = mlp_model.predict(X_test_scaled)
mlp_metrics = compute_effort_metrics(y_test, mlp_pred)

print('Neural Network Results:')
for metric, value in mlp_metrics.items():
    print(f'  {metric}: {value:.4f}')

## 9. Ensemble Methods

### 9.1 Voting Regressor

In [ ]:
# Voting Regressor (average predictions from multiple models)
voting_model = VotingRegressor([
    ('rf', rf_model),
    ('gb', gb_model),
    ('xgb', xgb_model),
    ('lgb', lgb_model),
    ('et', et_model)
])

voting_model.fit(X_train_scaled, y_train)
voting_pred = voting_model.predict(X_test_scaled)
voting_metrics = compute_effort_metrics(y_test, voting_pred)

print('Voting Regressor Results:')
for metric, value in voting_metrics.items():
    print(f'  {metric}: {value:.4f}')

### 9.2 Stacking Regressor

In [ ]:
# Stacking Regressor (meta-learner on top of base models)
stacking_model = StackingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)),
        ('gb', GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)),
        ('xgb', xgb.XGBRegressor(n_estimators=100, random_state=RANDOM_STATE))
    ],
    final_estimator=Ridge(),
    cv=5
)

stacking_model.fit(X_train_scaled, y_train)
stacking_pred = stacking_model.predict(X_test_scaled)
stacking_metrics = compute_effort_metrics(y_test, stacking_pred)

print('Stacking Regressor Results:')
for metric, value in stacking_metrics.items():
    print(f'  {metric}: {value:.4f}')

## 10. Model Comparison

In [ ]:
# Create comparison DataFrame
results = pd.DataFrame({
    'Random Forest': rf_metrics,
    'Gradient Boosting': gb_metrics,
    'XGBoost': xgb_metrics,
    'LightGBM': lgb_metrics,
    'Extra Trees': et_metrics,
    'Neural Network': mlp_metrics,
    'Voting': voting_metrics,
    'Stacking': stacking_metrics
}).T

print('\nModel Comparison Summary:')
print('=' * 80)
print(results.round(4))

# Find best model for each metric
print('\nBest Models per Metric:')
print('-' * 80)
for col in results.columns:
    if col in ['R2', 'Pred25']:
        best = results[col].idxmax()
        print(f'{col}: {best} ({results.loc[best, col]:.4f})')
    else:
        best = results[col].idxmin()
        print(f'{col}: {best} ({results.loc[best, col]:.4f})')

## 11. Visualizations

### 11.1 Model Performance Comparison

In [ ]:
# Bar plots for key metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics_to_plot = ['MMRE', 'MAE', 'R2', 'Pred25']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    data = results[metric].sort_values(ascending=(metric not in ['R2', 'Pred25']))
    colors = sns.color_palette('husl', len(data))
    data.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
    ax.set_title(f'Model Comparison: {metric}', fontsize=12, fontweight='bold')
    ax.set_xlabel(metric)
    ax.set_ylabel('Model')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 11.2 Feature Importance

In [ ]:
# Feature importance from best tree-based model
best_model = xgb_model  # Change to your best performing model

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1][:10]  # Top 10
    
    plt.figure(figsize=(12, 6))
    plt.title('Top 10 Feature Importances', fontsize=14, fontweight='bold')
    plt.barh(range(len(indices)), importances[indices], color='steelblue', edgecolor='black')
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel('Importance Score')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('Selected model does not have feature_importances_ attribute')

### 11.3 Prediction vs Actual

In [ ]:
# Scatter plots for best models
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

predictions = {
    'Random Forest': rf_pred,
    'XGBoost': xgb_pred,
    'LightGBM': lgb_pred,
    'Neural Network': mlp_pred,
    'Voting': voting_pred,
    'Stacking': stacking_pred
}

for idx, (name, pred) in enumerate(predictions.items()):
    ax = axes[idx]
    ax.scatter(y_test, pred, alpha=0.6, edgecolors='black', s=60)
    
    # Perfect prediction line
    min_val = min(y_test.min(), pred.min())
    max_val = max(y_test.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    
    r2 = r2_score(y_test, pred)
    ax.set_title(f'{name}\nR² = {r2:.4f}', fontweight='bold')
    ax.set_xlabel('Actual Effort')
    ax.set_ylabel('Predicted Effort')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 11.4 Residual Analysis

In [ ]:
# Residual plots for best model
best_pred = stacking_pred  # Change to your best model
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(best_pred, residuals, alpha=0.6, edgecolors='black')
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_title('Residual Plot', fontweight='bold')
axes[0].set_xlabel('Predicted Effort')
axes[0].set_ylabel('Residuals')
axes[0].grid(True, alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=15, color='steelblue', edgecolor='black')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 12. Cross-Validation Analysis

In [ ]:
# Cross-validation with K-Fold
cv_results = {}
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=RANDOM_STATE, verbose=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)
}

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='r2', n_jobs=-1)
    cv_results[name] = {
        'mean': scores.mean(),
        'std': scores.std(),
        'scores': scores
    }
    print(f'{name}: R² = {scores.mean():.4f} (+/- {scores.std():.4f})')

# Visualize CV results
plt.figure(figsize=(12, 6))
bp = plt.boxplot([cv_results[name]['scores'] for name in models.keys()],
                 labels=models.keys(), patch_artist=True)

for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
    patch.set_edgecolor('black')

plt.title('Cross-Validation R² Scores (5-Fold)', fontsize=14, fontweight='bold')
plt.ylabel('R² Score')
plt.xticks(rotation=15)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 13. Summary and Conclusions

### Key Improvements Implemented:

1. **Advanced ML Models**: XGBoost, LightGBM, Gradient Boosting, Neural Networks
2. **Feature Engineering**: Polynomial features, log transformations
3. **Robust Scaling**: Better handling of outliers
4. **Ensemble Methods**: Voting and Stacking regressors for improved predictions
5. **Comprehensive Metrics**: MMRE, MdMRE, Pred25, MAE, RMSE, MAPE, R²
6. **Cross-Validation**: K-Fold validation for model robustness
7. **Rich Visualizations**: Feature importance, residual analysis, model comparison

### Best Practices:
- Use ensemble methods for better generalization
- Apply robust scaling for datasets with outliers
- Monitor multiple metrics (not just R²)
- Validate models with cross-validation
- Analyze residuals to check model assumptions

## 14. Save Best Model (Optional)

Uncomment to save the best model for future use.

In [ ]:
# import pickle
# 
# # Save best model
# with open('best_cocomo_model.pkl', 'wb') as f:
#     pickle.dump(stacking_model, f)
# 
# # Save scaler
# with open('scaler.pkl', 'wb') as f:
#     pickle.dump(scaler, f)
# 
# print('Model and scaler saved successfully!')